# Triplet Loss: iResNet-34 face embeddings

**Flow: image processing → model training → loss graphs and saved artifacts.**

Use the **Python (mtrlearn CUDA)** kernel. Both experiments use random iResNet-34, 112×112 RGB crops and 256-dimensional normalized embeddings. No pretrained face weights are downloaded. Full runs can take many hours. Set `SMOKE_TEST=True` for a bounded real-image code check; its outputs are isolated under `artifacts/smoke/`.

## 1. Environment and experiment settings

Create the environment once with `conda env create -f environment.yml`, then register the kernel with `python -m ipykernel install --user --name mtrlearn --display-name "Python (mtrlearn CUDA)"` inside the activated environment. Do not install packages into a running notebook kernel.

In [ ]:
from pathlib import Path
import os
import sys
import json

ROOT = Path.cwd().resolve()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "recognition").is_dir():
        ROOT = candidate
        break
else:
    raise RuntimeError("Open this notebook from the repository or notebooks directory.")
sys.path.insert(0, str(ROOT))

import torch
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as DisplayImage
from src.recognition.config import ExperimentConfig
from src.recognition.data import prepare_manifests, FaceDataset, PKBatchSampler, make_loader
from src.recognition.models import IResNet34
from src.recognition.losses import TripletMetricLoss, ArcFaceHead
from src.recognition.training import configure_runtime, run_sweep, evaluate_selected
from src.recognition.notebook_utils import processing_preview, pilot_runtime, smoke_prepared, smoke_config

print("Python:", sys.executable)
print("PyTorch:", torch.__version__, "CUDA runtime:", torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError("Select the mtrlearn CUDA kernel before running training.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
SMOKE_TEST = os.environ.get("MTRLEARN_SMOKE", "0") == "1"
RUN_FINAL_TEST = False  # Enable only after the margin-selection design is fixed.

CONFIG = ExperimentConfig(
    method="triplet",
    data_root=ROOT / "data" / "VGGFace2",
    artifacts_root=ROOT / "artifacts",
    identities_per_batch=8,
    images_per_identity=4,
    epochs=60,
    num_workers=0 if os.name == "nt" else 2,
    device="cuda",
    amp=True,
)
configure_runtime(CONFIG)
display(pd.Series(CONFIG.to_dict(), name="Experiment settings"))

## 2. Image processing and identity-disjoint manifests

Verify the source catalog, hash exact duplicates, and keep one representative without deleting originals. Split original train identities into 432 train and 48 dev; original val identities become the 60-person final test. The two notebooks reuse the same seed-42 manifests, evaluation pairs and training probe. Near duplicates and identity aliases still require review; byte hashing cannot detect them.

In [ ]:
prepared = prepare_manifests(
    CONFIG.data_root,
    ROOT / "artifacts" / "shared" / "manifests",
    split_seed=42,
    dev_identities=48,
    genuine_per_identity=200,
    impostor_per_identity_pair=100,
    probe_per_identity=4,
)
if SMOKE_TEST:
    prepared = smoke_prepared(prepared)
    CONFIG = smoke_config(CONFIG)
    print("SMOKE TEST: tiny real-image subsets, altered curriculum gate, bounded epochs, no accuracy claims.")

display(pd.DataFrame([
    {"split": name, "identities": frame.identity_id.nunique(), "images": len(frame)}
    for name, frame in [("train", prepared.train), ("dev", prepared.dev), ("test", prepared.test)]
]))
print("Dev pairs:", len(prepared.dev_pairs), "Test pairs:", len(prepared.test_pairs))
print("Training probe triplets:", len(prepared.probe_triplets))

In [ ]:
figure = processing_preview(prepared, CONFIG.data_root, CONFIG.output_dir / "graphs" / "image_processing.png")
display(figure)
plt.close(figure)

sampler = PKBatchSampler(prepared.train.label.to_numpy(), P=CONFIG.identities_per_batch,
                         K=CONFIG.images_per_identity, batches_per_epoch=1, seed=42)
batch = next(iter(make_loader(prepared.train, CONFIG.data_root, train=True,
                              batch_sampler=sampler, num_workers=0, seed=42)))
print("Image tensor:", tuple(batch["image"].shape), batch["image"].dtype)
print("Normalized range:", float(batch["image"].min()), float(batch["image"].max()))
print("Images per identity:", torch.unique(batch["label"], return_counts=True)[1].tolist())
del batch

## 3. Model and objective

The reusable source is in `src/recognition/models.py` and `losses.py`. The model uses iResNet stages [3,4,6,3], a 256-feature linear layer and final BatchNorm, followed by explicit FP32 L2 normalization. Each margin run starts from the same encoder initialization for its seed.

In [ ]:
model = IResNet34(embedding_dim=256)
print(model)
print("Total encoder parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Trainable encoder parameters:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
del model

B0 uses squared Euclidean Triplet Loss on unit embeddings, margins **0.2, 0.3, 0.4**. All B(K−1) directed positive pairs form the denominator, including missing semi-hard candidates as zero. Random negatives are used first. After at least five random epochs, three consecutive fixed TRAIN probe checks must pass ranking rate ≥0.75, mean distance gap ≥0.05 and feature spread >0.001. The switch is one-way. Strict semi-hard selection has no hard-negative fallback; empty batches skip optimizer updates.

In [ ]:
objective = TripletMetricLoss(margin=0.2)
print(objective)
print("Margin sweep:", CONFIG.margins)
print("Squared distance: d2(a,p) = ||normalize(a) - normalize(p)||^2")

## 4. Common GPU memory pilot

Measure two bounded optimization steps for both objectives using disposable models, then choose a common batch profile. The preferred profile is P=8, K=4, B=32; if it runs out of GPU memory, both use P=4, K=4, B=16. The pilot never supplies learned weights to the experiment. Close other GPU jobs before running it.

In [ ]:
CONFIG = pilot_runtime(CONFIG, prepared)
print("Selected common profile:", CONFIG.identities_per_batch, "x", CONFIG.images_per_identity,
      "=", CONFIG.batch_size)
display(pd.Series(json.loads((Path(CONFIG.artifacts_root) / "shared" / "runtime_pilot.json").read_text())))

## 5. Model training and margin sweep

AdamW peak LR 3e-4, five-epoch warmup, cosine decay, maximum 60 epochs and dev-only early stopping. Three margins are screened at seed 42; the selected margin is repeated at seeds 123 and 2026. Completed runs are reused; interrupted runs resume at the last completed epoch. Change the artifact root when changing scientific settings; mismatched resume configurations are rejected. The final test is not used during this cell.

In [ ]:
sweep_val = run_sweep(CONFIG, prepared)
display(sweep_val[["method", "margin", "seed", "stage", "best_epoch", "dev_tar", "dev_auc", "dev_eer"]])
print("Saved sweep:", CONFIG.output_dir / "sweep_val.csv")

## 6. Loss graphs and saved artifacts

Each run saves `history.csv`, loss/verification graphs, resumable `best.pt` and `last.pt`, and a standalone `embedding.pt` with transform metadata and a dev-calibrated threshold. The experiment folder saves `sweep_val.csv`, `selection.json` and the margin-comparison graph. Smoke-test outputs must never be reported as trained-model performance.

In [ ]:
display(DisplayImage(filename=str(CONFIG.output_dir / "graphs" / "sweep_val.png")))
for run in sweep_val.itertuples():
    history_path = Path(run.history_path)
    history = pd.read_csv(history_path)
    display(history.tail())
    display(DisplayImage(filename=str(history_path.parent / "graphs" / "loss.png")))
print("Artifact root:", CONFIG.output_dir)
print("Weights:", *sorted(str(path) for path in CONFIG.output_dir.glob("runs/*/weights/embedding.pt")), sep="\n")

## 7. Final test after dev selection

Set `RUN_FINAL_TEST=True` only after fixing the selected margin/protocol. This evaluates only the selected margin, using each seed's unchanged dev threshold. Open-set gallery/probe protocols are shared, with a separate dev-calibrated search threshold. The training-class head is never used for unseen-person verification.

In [ ]:
if RUN_FINAL_TEST:
    test_results = evaluate_selected(CONFIG, prepared, sweep_val)
    display(test_results)
    print("Saved:", CONFIG.output_dir / "test_results.csv")
else:
    print("Final test remains unused. Set RUN_FINAL_TEST=True after completing dev selection.")

References: [iResNet source](https://github.com/deepinsight/insightface/blob/master/recognition/arcface_torch/backbones/iresnet.py), [FaceNet](https://arxiv.org/abs/1503.03832), [ArcFace](https://arxiv.org/abs/1801.07698), [OpenMMLab angular fallback](https://github.com/open-mmlab/mmpretrain/blob/main/mmpretrain/models/heads/margin_head.py).